# Task 4: Context-Aware Chatbot Using LangChain + RAG

In [ ]:
!pip install langchain sentence-transformers faiss-cpu gradio pypdf openai -q

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader

loader = TextLoader('sample_data.txt')
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)
len(docs)


In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embedding_model)

retriever = vectorstore.as_retriever()


In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)


In [ ]:
from langchain.llms import OpenAI
from langchain.chains import ConversationalRetrievalChain

# Add your OpenAI API key
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

llm = OpenAI(temperature=0)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)


In [ ]:
query = "What is this document about?"
response = qa_chain.run(query)

print(response)


In [ ]:
import gradio as gr

def chatbot(message, history):
    response = qa_chain.run(message)
    return response

iface = gr.ChatInterface(fn=chatbot, title="RAG Chatbot")

iface.launch()
